# Predict Chemical Reaction

## SMILES -> Graph

In [8]:
%pip install rdkit
%pip install torch_geometric

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit.Chem import PandasTools
import pandas as pd
from rdkit import Chem
import torch
from torch_geometric.data import Data

In [13]:
def smiles_to_mol(smiles):
    """
    Convert a SMILES string into an RDKit Mol object.
    Returns None if the SMILES string is invalid.
    """
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None

    return mol


def print_mol_basic_info(smiles):
    """
    Print basic RDKit molecule information.
    """
    mol = smiles_to_mol(smiles)

    if mol is None:
        return

    canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

    print("Original SMILES:", smiles)
    print("Canonical SMILES:", canonical_smiles)
    print("Number of atoms:", mol.GetNumAtoms())
    print("Number of bonds:", mol.GetNumBonds())

    return mol

In [14]:
reactant_1 = "CCO"    # ethanol
reactant_2 = "O=O"    # oxygen

mol_1 = print_mol_basic_info(reactant_1)
print()
mol_2 = print_mol_basic_info(reactant_2)

Original SMILES: CCO
Canonical SMILES: CCO
Number of atoms: 3
Number of bonds: 2

Original SMILES: O=O
Canonical SMILES: O=O
Number of atoms: 2
Number of bonds: 1


## Data Stucture
- Atoms
- Edges_index (Atoms - Atoms)
- Edges_attr - bond feature matrix

In [ ]:
import torch
from torch_geometric.data import Data
from rdkit import Chem


def atom_features(atom):
    return [
        atom.GetAtomicNum(),          # Atom number: C=6, O=8, N=7
        atom.GetDegree(),             # Number of connected atoms
        atom.GetFormalCharge(),       # Formal charge
        int(atom.GetIsAromatic())     # Aromaticity
    ]


def bond_features(bond):
    bond_type = bond.GetBondType()

    return [
        int(bond_type == Chem.rdchem.BondType.SINGLE),
        int(bond_type == Chem.rdchem.BondType.DOUBLE),
        int(bond_type == Chem.rdchem.BondType.TRIPLE),
        int(bond_type == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing())
    ]


def smiles_to_pyg_data(smiles, y=None):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    # 1. Atom features
    x = []
    for atom in mol.GetAtoms():
        x.append(atom_features(atom))

    x = torch.tensor(x, dtype=torch.float)

    # 2. Edge index + edge attributes
    edge_index = []
    edge_attr = []

    for bond in mol.GetBonds():
        start = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()

        bond_feat = bond_features(bond)

        # Usually, PyG uses undirected graphs by adding edges in both directions
        edge_index.append([start, end])
        edge_index.append([end, start])

        edge_attr.append(bond_feat)
        edge_attr.append(bond_feat)

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)

    return data

/Users/DeMo/miniconda3/envs/274P/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
data = smiles_to_pyg_data("CCO")

print(data)
print(data.x)
print(data.edge_index)
print(data.edge_attr)

Data(x=[3, 4], edge_index=[2, 4], edge_attr=[4, 6])
tensor([[6., 1., 0., 0.],
        [6., 2., 0., 0.],
        [8., 1., 0., 0.]])
tensor([[0, 1, 1, 2],
        [1, 0, 2, 1]])
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.]])
